# Unmix the Signal — ICA from scratch
Write-up notebook. Run from the repo root (`jupyter notebook notebooks/report.ipynb`).

## 1. Setup

In [ ]:
import sys, glob, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from unmix import *
from unmix.evaluation import corr_matrix
from unmix.preprocessing import center, whiten
from IPython.display import Audio, display
paths = sorted(glob.glob('../data/sources/*.wav'))
S, sr = load_sources(paths, max_seconds=10)
names = [p.split('/')[-1][:-4] for p in paths]
N = len(paths); print(N, 'sources', S.shape, sr)

## 2. Mix

In [ ]:
A = make_mixing_matrix(N, seed=0)
X = add_noise(mix(S, A), snr_db=30)
print('A =\n', A.round(2), '\ncond(A) =', np.linalg.cond(A).round(2))

## 3. Center + PCA whitening
Covariance of whitened data should be the identity.

In [ ]:
Xc, mean = center(X)
Z, V, V_inv, eig = whiten(Xc)
print(np.cov(Z).round(6))

## 4. FastICA + PCA baseline

In [ ]:
res = ica_separate(X, N, fun='logcosh')
print('iterations:', res['n_iter'])
ev_ica = evaluate(res['S_hat'], S)
ev_pca = evaluate(pca_separate(X, N), S)
print('ICA', ev_ica['mean_corr'], ev_ica['mean_sdr_db'])
print('PCA', ev_pca['mean_corr'], ev_pca['mean_sdr_db'])

## 5. Plots

In [ ]:
from unmix.plotting import plot_waveforms, plot_corr_matrices
plot_waveforms(S, X, ev_ica['aligned'], sr, names, '../outputs/plots/waveforms.png')
plot_corr_matrices(corr_matrix(pca_separate(X,N), S), corr_matrix(res['S_hat'], S), names, '../outputs/plots/correlation_pca_vs_ica.png')
plt.figure(figsize=(12,6)); plt.imshow(plt.imread('../outputs/plots/waveforms.png')); plt.axis('off'); plt.show()

In [ ]:
for j,n in enumerate(names):
    print(n); display(Audio(ev_ica['aligned'][j], rate=sr))

## 6. Experiment: SNR sweep
Run `python scripts/snr_sweep.py`, then view `outputs/plots/snr_sweep.png`.

In [ ]:
plt.figure(figsize=(10,4)); plt.imshow(plt.imread('../outputs/plots/snr_sweep.png')); plt.axis('off'); plt.show()

## 7. Discussion
* **ICA vs PCA:** …
* **Whitening:** …
* **Non-Gaussianity / CLT:** …
* **Why these metrics/experiments:** …
* **Limitations:** instantaneous mixing only, ≤1 Gaussian source.